# Load dependencies

In [ ]:
import matplotlib.pyplot as plt, torch
from torch.distributions.multivariate_normal import MultivariateNormal
from torch import tensor

In [ ]:
torch.manual_seed(42)
torch.set_printoptions(precision=3, linewidth=140, sci_mode=False)

# Create data

In [ ]:
n_clusters=6
n_samples =250
synthetic_data_mean = 30
synthetic_data_std = 50
center_cov_mat = torch.diag(tensor([5.,5.]))

centroids = (torch.rand(n_clusters, 2) - 0.5) * synthetic_data_std + synthetic_data_mean

def sample(m):
  return MultivariateNormal(m, center_cov_mat).sample((n_samples,))

slices = [sample(c) for c in centroids]
data = torch.cat(slices)

# Inefficient implementation

In [ ]:
X = data.clone()
n_cluster = 6
# Select k random elements along axis 0
indices = torch.randint(high=X.shape[0], size=(n_cluster,))
current_cents = X[indices]
print(f"current_cents: {current_cents}")

n_cluster=current_cents.shape[0]
# update centroid assignment
cent_map_pt = torch.zeros(X.shape[0])
for i, x in enumerate(X):
    dist = ((x-current_cents)**2).sum(-1)#.sqrt() # Broadcast
    cent_map_pt[i] = dist.argmin(0)

# update centroid coordinates
new_cents = torch.stack([
    X[cent_map_pt == i].mean(axis=0)
    for i in range(n_cluster)
])
print(f"new centroids: {new_cents}")

In [ ]:
def one_update(X, current_cents):
    n_cluster=current_cents.shape[0]
    # update centroid assignment
    cent_map_pt = torch.zeros(X.shape[0])
    for i, x in enumerate(X):
        dist = ((x-current_cents)**2).sum(-1)#.sqrt() # Broadcast
        cent_map_pt[i] = dist.argmin(0)

    # update centroid coordinates
    new_cent_list = []
    for i in range(n_cluster):
        new_cent = X[cent_map_pt == i].mean(axis=0)
        if new_cent.isnan().any(): new_cent = current_cents[i]
        new_cent_list.append(new_cent)
    new_cents = torch.stack(new_cent_list)
    return new_cents

print(one_update(X, current_cents))

In [ ]:
def kmeans(data, n_cluster=6, n_iter=5, return_history=False):
    X = data.clone()
    indices = torch.randint(high=X.shape[0], size=(n_cluster,))
    current_cents = X[indices]

    history = [current_cents.clone()] if return_history else None
    for _ in range(n_iter):
        current_cents = one_update(X, current_cents)
        if return_history:
            history.append(current_cents.clone())

    if return_history:
        history = torch.stack(history) # [T, K, 2]
        return (current_cents, history)
    else:
        return current_cents
     
%timeit -n 10 centroids=kmeans(data)

# Plot data

In [ ]:
def plot_data(centroids, data, n_samples, ax=None):
    if ax is None: _,ax = plt.subplots()
    for i, centroid in enumerate(centroids):
        samples = data[i*n_samples:(i+1)*n_samples]
        ax.scatter(samples[:,0], samples[:,1], s=1)
        ax.plot(*centroid, markersize=10, marker="x", color='k', mew=5)
        ax.plot(*centroid, markersize=5, marker="x", color='m', mew=2)

In [ ]:
centroids=kmeans(data)
plot_data(centroids, data, n_samples)

# Animation

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def animate_kmeans(X, cent_history, interval=600):
    X_np = X.clone().cpu().numpy()
    H = cent_history.clone().cpu().numpy()
    K = H.shape[1]

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(X_np[:, 0], X_np[:, 1], s=20, c="lightgray", alpha=0.8)
    cents_sc = ax.scatter([], [], s=180, c="red", marker="X", edgecolor="black")

    # trajectory line: from t-1 -> t
    trail_lines = [ax.plot([], [], '-', lw=2, alpha=0.8)[0] for _ in range(K)]

    ax.set_xlim(X_np[:, 0].min() - 1, X_np[:, 0].max() + 1)
    ax.set_ylim(X_np[:, 1].min() - 1, X_np[:, 1].max() + 1)

    def update(t):
        C = H[t]
        cents_sc.set_offsets(C)

        for j in range(K):
            if t == 0:
                trail_lines[j].set_data([], [])
            else:
                seg = H[:t+1, j, :]
                trail_lines[j].set_data(seg[:, 0], seg[:, 1])

        ax.set_title(f"KMeans centroid evolution - iter {t}")
        return (cents_sc, *trail_lines)

    ani = FuncAnimation(fig, update, frames=H.shape[0], interval=interval, blit=True, repeat=False)
    plt.close(fig)
    return ani

In [ ]:
n_paths = 1
final_cents_list = []
hist_list = []
for _ in range(n_paths):
    final_cents, hist = kmeans(data, n_cluster=6, n_iter=5, return_history=True)
    final_cents_list.append(final_cents)
    hist_list.append(hist)

In [ ]:
hist = torch.stack(hist_list).mean(0)
final_cents = torch.stack(final_cents_list).mean(0)
# this approach apparently doesn't make it convergent.
# it only biases the centroids heavily toward the center of the whole distribution.
# TODO: i think the aggregation should be per point (over 1500 points) not paths.
ani = animate_kmeans(data, hist, interval=400)
HTML(ani.to_jshtml())